## Aeropulse — Silver: Airport

**Purpose:** Bronze → Silver for the airport reference table. Cleans, standardises and splits `airport_description` into `airport_name` / `airport_city` / `airport_state`, then merges into `silver.airport`.

**Load type:** Full refresh (bronze_airport is fully overwritten on every ingestion run, so this always processes the complete reference set — no batch-window filtering needed).

**Depends on:** `silver-environment`, `silver-helper` (run via `%run`)

**Reads:** `aeropulse_bronze_lh.dbo.bronze_airport` (via `airport_bronze_path`)

**Writes:** `silver.airport` (merge on `airport_code`)

**Default lakehouse:** `aeropulse_silver_lh`


In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

StatementMeta(, 14879790-a15c-45c3-a5f6-e76d53ba9809, 3, Finished, Available, Finished, False)

In [2]:
batch_id = ""
batch_year = ""

StatementMeta(, 14879790-a15c-45c3-a5f6-e76d53ba9809, 4, Finished, Available, Finished, False)

In [3]:
%run silver-environment

StatementMeta(, 14879790-a15c-45c3-a5f6-e76d53ba9809, 5, Finished, Available, Finished, True)

In [4]:
%run silver-helper

StatementMeta(, 14879790-a15c-45c3-a5f6-e76d53ba9809, 8, Finished, Available, Finished, True)

In [5]:
airport_df = spark.read.format('delta').load(airport_bronze_path).filter(F.col("batch_id") == batch_id)

StatementMeta(, 14879790-a15c-45c3-a5f6-e76d53ba9809, 9, Finished, Available, Finished, False)

In [6]:
# Data cleaning/transformation & standardisation

# dict: for rename mapping
rename_mapping = {
    "Code":"airport_code",
    "Description":"airport_description"
}
/
# function: rename_column headers
airport_df = (
    airport_df
    .transform(
        trim_whitespaces)
        .transform(lambda d: rename_column(d, rename_mapping))
)

# remove nulls using remove_nulls function
airport_df = remove_nulls(airport_df, ["airport_code"])

# create custom columns, airport_name, airport_city, airport_state
airport_df = (airport_df
    .withColumn("_name_split", F.split(F.col("airport_description"), ": ", 2))
    .withColumn("airport_name", F.trim(F.col("_name_split").getItem(1)))
    .withColumn("_location_split", F.split(F.col("_name_split").getItem(0), ", ", 2))
    .withColumn("airport_city", F.trim(F.col("_location_split").getItem(0)))
    .withColumn("airport_state", F.trim(F.col("_location_split").getItem(1)))
    .drop("_name_split", "_location_split")
)

# remove duplicates
airport_df = remove_duplicates(airport_df, ["airport_code"])

# create surrogate key
airport_df = add_sk_key(airport_df, ["airport_code"], "airport_sk")





StatementMeta(, 14879790-a15c-45c3-a5f6-e76d53ba9809, 10, Finished, Available, Finished, False)

In [7]:
# write to silver layer

update_cols = [c for c in airport_df.columns if c not in ["airport_code"]]

write_to_silver(
    airport_df,
    "silver.airport",
    "s.airport_code = t.airport_code",
    update_cols
)

StatementMeta(, 14879790-a15c-45c3-a5f6-e76d53ba9809, 11, Finished, Available, Finished, False)